# Embedding-based KNN Router
Uses pre-trained sentence embeddings and nearest neighbors to route queries.


In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
import json

train = pd.read_csv(f"{DATA_DIR}/train.csv")
models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]
max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()

rewards = pd.DataFrame(index=train.index, columns=models)
for m in models:
    p = train[f"{m}_performance"]
    c = train[f"{m}_cost"]
    rewards[m] = 0.85 * p - 0.15 * (c / global_avg_max_cost)

In [3]:
# Embed Queries
# Note: Ensure you have internet access or pre-downloaded the model
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# embeddings = embedder.encode(train['query'].tolist(), show_progress_bar=True)

print("Loading cached Qwen 8B embeddings...")
embeddings = np.load(f"{CACHE_DIR}/dense_Qwen-Qwen3-Embedding-8B_features.npy")

Loading cached Qwen 8B embeddings...


In [ ]:
# K-Fold CV for KNN Routing
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = []
k = 1000

for train_idx, val_idx in kf.split(embeddings):
    X_train, X_val = embeddings[train_idx], embeddings[val_idx]
    
    nn = NearestNeighbors(n_neighbors=k, metric='cosine')
    nn.fit(X_train)
    
    distances, indices = nn.kneighbors(X_val)
    
    # === AVERAGE DISTANCE === 
    # for i, neighbors in enumerate(indices):
    #     # average reward of neighbors for each model
    #     neighbor_real_idx = train_idx[neighbors]
    #     avg_neighbor_rewards = rewards.iloc[neighbor_real_idx].mean(axis=0)
    #     best_model = avg_neighbor_rewards.idxmax()
    #     oof_preds.append((val_idx[i], best_model))
    
    # === COSINE SIMILARITY ===
    for i, (neighbors, dists) in enumerate(zip(indices, distances)):
        neighbor_real_idx = train_idx[neighbors]
        
        # Cosine distance is (1 - similarity). Convert distances back to similarities
        similarities = 1.0 - dists
        similarities = np.clip(similarities, 1e-5, 1.0) # Avoid division by zero
        
        # Normalize similarities to sum to 1.0
        weights = similarities / np.sum(similarities)
        
        # Compute the weighted average of the neighbors' rewards
        neighbor_rewards = rewards.iloc[neighbor_real_idx].values
        weighted_rewards = np.dot(weights, neighbor_rewards) # Shape: (11,)
        
        best_model = models[np.argmax(weighted_rewards)]
        oof_preds.append((val_idx[i], best_model))

oof_preds.sort(key=lambda x: x[0])
train['oof_pred_model_knn'] = [p[1] for p in oof_preds]

In [ ]:
# Calculate CV Reward
pred_p = []
pred_c = []
for i, row in train.iterrows():
    pred_m = row['oof_pred_model_knn']
    pred_p.append(row[f"{pred_m}_performance"])
    pred_c.append(row[f"{pred_m}_cost"])

avg_p = np.mean(pred_p)
avg_c = np.mean(pred_c)
cv_reward = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)

print("CV Reward 0.85 (KNN Routing):", cv_reward)
print("CV Average Performance:", avg_p)
print("CV Average Cost:", avg_c)

In [ ]:
# Log to experiment_summary.csv
summary = pd.read_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv")
new_row = {
    'Experiment_ID': f'KNN_MiniLM_k{k}',
    'Method_Category': 'KNN',
    'Features': 'Qwen3-embedding-8b',
    'K-Fold': k,
    'CV_Reward_0.85': round(cv_reward, 4),
    'CV_Avg_Performance': round(avg_p, 4),
    'CV_Avg_Cost': round(avg_c, 4),
    'Model_Distribution': str(train['oof_pred_model_knn'].value_counts().to_dict()),
    'Public_Kaggle_Score': None
}
summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
summary.to_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv", index=False)

In [4]:
# === ENSEMBLE: RIDGE REGRESSION + KNN ===
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors

# 1. Setup Data & Targets
kf = KFold(n_splits=100, shuffle=True, random_state=42)
oof_preds = []

# Assuming 'embeddings' and 'rewards' are already loaded in memory from earlier cells
y_reg = rewards.values
ridge_alpha = 1.0
knn_k = 50
ensemble_weight = 0.5  # 0.5 means 50% Ridge, 50% KNN

print(f"Running Ensemble (Ridge weight: {ensemble_weight}, KNN weight: {1 - ensemble_weight})...")

# 2. Cross-Validation Loop
for train_idx, val_idx in kf.split(embeddings):
    X_train, X_val = embeddings[train_idx], embeddings[val_idx]
    
    # --- RIDGE PREDICTIONS ---
    ridge_fold_preds = np.zeros((len(val_idx), len(models)))
    for m_idx in range(len(models)):
        y_train = y_reg[train_idx, m_idx]
        reg = Ridge(alpha=ridge_alpha)
        reg.fit(X_train, y_train)
        ridge_fold_preds[:, m_idx] = reg.predict(X_val)
        
    # --- KNN PREDICTIONS ---
    nn = NearestNeighbors(n_neighbors=knn_k, metric='cosine')
    nn.fit(X_train)
    distances, indices = nn.kneighbors(X_val)
    
    knn_fold_preds = np.zeros((len(val_idx), len(models)))
    for i, (neighbors, dists) in enumerate(zip(indices, distances)):
        neighbor_real_idx = train_idx[neighbors]
        
        # Weighted KNN based on cosine similarity
        similarities = np.clip(1.0 - dists, 1e-5, 1.0)
        weights = similarities / np.sum(similarities)
        
        neighbor_rewards = rewards.iloc[neighbor_real_idx].values
        knn_fold_preds[i, :] = np.dot(weights, neighbor_rewards)
        
    # --- BLEND PREDICTIONS ---
    blended_preds = (ensemble_weight * ridge_fold_preds) + ((1.0 - ensemble_weight) * knn_fold_preds)
    
    # Argmax to find the best model for each query in the validation fold
    for i in range(len(val_idx)):
        best_model = models[np.argmax(blended_preds[i])]
        oof_preds.append((val_idx[i], best_model))

# 3. Evaluate Results
oof_preds.sort(key=lambda x: x[0])
train['oof_pred_model_ensemble'] = [p[1] for p in oof_preds]

pred_p = [train.loc[i, f"{m}_performance"] for i, m in enumerate(train['oof_pred_model_ensemble'])]
pred_c = [train.loc[i, f"{m}_cost"] for i, m in enumerate(train['oof_pred_model_ensemble'])]

avg_p = np.mean(pred_p)
avg_c = np.mean(pred_c)
cv_reward = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)

print(f"\n[+] Ensemble CV Reward 0.85: {cv_reward:.5f}")
print(f"[+] Ensemble Average Performance: {avg_p:.4f}")
print(f"[+] Ensemble Average Cost: {avg_c:.4f}")

# 4. Optional: Log to experiment_summary.csv
summary = pd.read_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv")
new_row = {
    'Experiment_ID': f'Ensemble_Ridge_KNN_w{ensemble_weight}',
    'Method_Category': 'Ensemble',
    'Features': 'Qwen3-embedding-8b',
    'K-Fold': 15,
    'CV_Reward_0.85': round(cv_reward, 4),
    'CV_Avg_Performance': round(avg_p, 4),
    'CV_Avg_Cost': round(avg_c, 4),
    'Model_Distribution': str(train['oof_pred_model_ensemble'].value_counts().to_dict()),
    'Public_Kaggle_Score': None
}
summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
summary.to_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv", index=False)


Running Ensemble (Ridge weight: 0.5, KNN weight: 0.5)...

[+] Ensemble CV Reward 0.85: 0.47479
[+] Ensemble Average Performance: 0.5891
[+] Ensemble Average Cost: 0.0134


/tmp/ipykernel_2331595/218836185.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
